## Exploring predictions

In [1]:
import os
import sys
import importlib as imp
import numpy as np
import matplotlib.pyplot as plt
import torch

import experiment_config
import build_model
import build_data
import read_landsat
import methods
import methods
import predictions
import read_landsat

import utils

from sklearn.metrics import mean_squared_error, mean_absolute_error

ModuleNotFoundError: No module named 'tensorflow'

In [2]:
print(f"python version = {sys.version}")
print(f"numpy version = {np.__version__}")
print(f"pytorch version = {torch.__version__}")

python version = 3.12.0 | packaged by conda-forge | (main, Oct  3 2023, 08:36:57) [Clang 15.0.7 ]
numpy version = 1.26.4
pytorch version = 2.1.2.post3


In [ ]:
# GET config
EXP_NAME = "exp7"
REWRITE = False

# config = experiment_config.get_config(EXP_NAME)
config = utils.get_config("EXP_NAME")
config["batch_size"] = 256
config["mode"] = "inference"

directory_paths = methods.get_directories()
SAVE_MODEL_DIRECTORY = directory_paths["save_model_dir"]
DATA_DIRECTORY = directory_paths["data_dir"]
FIGURE_DIRECTORY = directory_paths["figures_dir"]
PREDICTIONS_DIRECTORY = directory_paths["predictions_dir"]
LANDSAT_DIRECTORY = directory_paths["landsat_dir"]
MOSAICS_DIRECTORY = directory_paths["mosaics_dir"]

In [ ]:
# GET THE DATA
imp.reload(build_data)
imp.reload(read_landsat)
imp.reload(methods)
imp.reload(predictions)

# for debugging
# config["inference_region"] = (-7, -6, 106, 107)
config["inference_region"] = (33, 35, 71, 72)

# set and get important config
(lat_s_bound, lat_n_bound, lon_w_bound, lon_e_bound) = read_landsat.get_landsat_bounds(
    config, region=config["inference_region"]
)

# load the model
model = build_model.get_model(config)

for year in np.arange(2020, 2023):  # 2000, 2005, 2007, 2010, 2021, 2022):
    print(" --- " + str(year) + "---")
    config["inference_years"] = (year,)
    filenames_list = []

    for latfile in np.arange(
        lat_s_bound + config["tile_len_deg"],
        lat_n_bound + config["tile_len_deg"],
        config["tile_len_deg"],
    ):
        for lonfile in np.arange(lon_w_bound, lon_e_bound, config["tile_len_deg"]):
            # CHECK IF LANDSAT FILE EXISTS
            config["tile"] = (
                latfile - config["tile_len_deg"],
                latfile,
                lonfile,
                lonfile + config["tile_len_deg"],
            )
            landsat_file = read_landsat.get_input_filename(
                config["inference_years"], (latfile,), (lonfile,), config
            )

            # CHECK IF LANDSAT FILE EXISTS
            if os.path.isfile(LANDSAT_DIRECTORY + landsat_file[0] + ".tif") is False:
                continue

            # TODO: check if landsat tile is all water, if so, create prediction file of all NODATA

            # CHECK IF PREDICTION FILE ALREADY EXISTS
            predictions_filename = (
                config["exp_name"] + "_predictions_" + landsat_file[0]
            )
            filenames_list.append(PREDICTIONS_DIRECTORY + predictions_filename + ".tif")
            if (
                os.path.isfile(PREDICTIONS_DIRECTORY + predictions_filename + ".tif")
                and REWRITE is False
            ):
                continue
            print(landsat_file[0])

            # GET THE SAMPLE TAGS
            tags_inf, __ = build_data.get_tags(config)
            if len(tags_inf[0]) == 0:
                continue

            # MAKE PREDICTIONS and SAVE AS TIF
            hfi_predict, hfi_labels, latlon_bounds = predictions.make_predictions(
                config, model, tags_inf
            )

            meta_data = predictions.save_predictions_tif(
                hfi_predict,
                PREDICTIONS_DIRECTORY + predictions_filename + ".tif",
                latlon_bounds=latlon_bounds,
            )

            filenames_list.append(PREDICTIONS_DIRECTORY + predictions_filename + ".tif")
            print("\n")

    # TILE THE PREDICTIONS TOGETHER
    mosaic_filename = (
        MOSAICS_DIRECTORY
        + config["exp_name"]
        + "_"
        + str(config["inference_years"][0])
        + "_mlhfi_mosaic.tif"
    )
    mosaic, mosaic_trans = predictions.create_mosaic(filenames_list)
    meta_data = predictions.save_predictions_tif(
        mosaic, mosaic_filename, trans=mosaic_trans
    )
    print("mosaic saved.")